# Tritium Detection with an Ionisation Chamber

A flow-through ionisation chamber measures the tritium concentration of a gas stream, for example the purge gas of a breeding blanket. The stream passes through the chamber unchanged, and the chamber produces a signal proportional to the tritium concentration.

The chamber receives the tritium flux $\Phi_{in}$ and the volumetric flow rate $\dot{V}$ of the stream. From these it computes the concentration and the signal

$$c = \frac{\Phi_{in}}{\dot{V}} \qquad \text{signal} = \varepsilon(c) \, c \qquad \Phi_{out} = \Phi_{in}$$

The detection efficiency $\varepsilon$ is either a constant or a function of the concentration. An optional detection limit $c_{min}$ sets the efficiency to zero below it:

$$\varepsilon(c) = 0 \quad \text{for} \quad c < c_{min}$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("../pathsim_docs.mplstyle")

from pathsim import Connection, Simulation
from pathsim.blocks import Constant, Scope, Source

from pathsim_fusion import IonisationChamber

## Tritium Release Pulse

The stream carries a tritium release pulse at a constant flow rate. Three chambers measure the same stream:

- an ideal chamber with $\varepsilon = 1$, its signal is the concentration itself
- a chamber with $\varepsilon = 0.8$ and a detection limit $c_{min} = 0.2$
- a chamber whose efficiency drops at high concentrations, $\varepsilon(c) = 1 / (1 + c / c_{sat})$

The values are illustrative and in arbitrary units.

In [ ]:
# tritium release pulse, peak flux 'phi_0' at 't_0'
phi_0, t_0 = 2.0, 5.0
flux = Source(lambda t: phi_0 * (t / t_0) * np.exp(1 - t / t_0))

# constant purge gas flow rate
flow = Constant(1.0)

# three chambers measuring the same stream
c_sat = 1.5
ideal = IonisationChamber()
limited = IonisationChamber(detection_efficiency=0.8, detection_threshold=0.2)
saturating = IonisationChamber(func_efficiency=lambda c: 1 / (1 + c / c_sat))

sco = Scope(labels=["ideal", "efficiency 0.8, limit 0.2", "saturating"])

chambers = [ideal, limited, saturating]

sim = Simulation(
    [flux, flow, *chambers, sco],
    [
        Connection(flux, *[ic["flux_in"] for ic in chambers]),
        Connection(flow, *[ic["flow_rate"] for ic in chambers]),
        *[Connection(ic["signal"], sco[i]) for i, ic in enumerate(chambers)],
    ],
    dt=0.05,
    log=True,
)

sim.run(40)

In [ ]:
sco.plot(lw=1.5)
plt.ylabel("signal")
plt.title("Ionisation chamber signals")
plt.show()

The chamber with the detection limit reads zero at the start of the pulse and again in its tail, where the concentration is below $c_{min}$. Above the limit its signal is 80 percent of the concentration. The saturating chamber follows the ideal signal at low concentrations and reads lower around the peak of the pulse.